In [13]:
import asyncio
import time
def make_coffee_sync():
    time.sleep(2)
async def make_coffee_async():
    await asyncio.sleep(2)
start = time.time()
make_coffee_sync()
make_coffee_sync()
print(time.time()-start)
async def main():
    start = time.time()
    await make_coffee_async()
    await make_coffee_async()
    print(time.time()-start)
await main()


4.000751733779907
4.021805286407471


In [18]:
import asyncio
import time
async def make_coffee_async(order_id):
    await asyncio.sleep(2)
async def main():
    start = time.time()
    results  = await asyncio.gather(
        make_coffee_async('A'),
        make_coffee_async('B'),
        make_coffee_async('C'),
    )
    print(time.time()-start)
await main()


2.0047247409820557


In [26]:
import asyncio
async def background_job(name:str,duration:float):
    await asyncio.sleep(duration)
async def main():
    task1 = asyncio.create_task(background_job('下载文件',2))
    task2 = asyncio.create_task(background_job('解析数据',1))
    task3 = asyncio.create_task(background_job('生成报告',3))
    await asyncio.sleep(0.5)
    results = await asyncio.gather(task1,task2,task3)
await main()

In [36]:
import asyncio
import time
async def limited_fetch(sem:asyncio.Semaphore,task_id:int) -> str:
    async with sem:
        await asyncio.sleep(1)
async def main():
    sem = asyncio.Semaphore(3)
    tasks = [limited_fetch(sem,i)for i in range(10)]
    start = time.time()
    results = await asyncio.gather(*tasks)
    elapsed = time.time()-start
    print(elapsed)
await main()

4.030964612960815


In [59]:
import asyncio
async def llm_stream_response(prompt:str):
    tokens = ["你好", "!", "你", "问", "的是", 
              f"「{prompt}」", "。", "这是", "流式", "回答", "。"]
    for token in tokens:
        await asyncio.sleep(0.1)
        yield token
async def main():
    prompt = '什么是异步编程?'
    async for token in llm_stream_response(prompt):
        print(token,end='',flush=True)
await main()

你好!你问的是「什么是异步编程?」。这是流式回答。

In [63]:
import asyncio
import random
async def producer(queue:asyncio.Queue,producer_id:int,num_items:int):
    for i in range(num_items):
        item = f'数据_{producer_id}_{i}'
        await asyncio.sleep(random.uniform(0.1,0.3))
        await queue.put(item)
    await queue.put(None)
async def consumer(queue:asyncio.Queue,consumer_id:int):
    processed = 0
    while True:
        item = await queue.get()
        if item is None:
            await queue.put(None)
            break
        await asyncio.sleep(0.2)
        processed += 1
async def main():
    queue = asyncio.Queue(maxsize=5)
    producers = [producer(queue,pid,4)for pid in range(2)]
    consumers = [consumer(queue,cid)for cid in range(2)]
    await asyncio.gather(*producers,*consumers)
await main()